# Predicting Breast Cancer using Breast Ultrasound Images

Έχουμε τώρα τα απαραίτητα εργαλεία να κοιτάξουμε ένα σημαντικό και δύσκολο πρόβλημα: την διάγνωση κακοήθη καρκίνου του μαστού, από υπέρηχο (ultrasound).

Η βασική μας προσέγγυση θα γίνει μέσω της ιδέας του **Transfer Learning** που είδαμε και στο [προηγούμενο Notebook](https://colab.research.google.com/drive/1tnIQoViTVbT-7XiJa791h492B_WuNw4t?usp=sharing).

Θα χρησιμοποιήσουμε ένα σετ δεδομένων από εδώ:

https://www.kaggle.com/datasets/sabahesaraki/breast-ultrasound-images-dataset

Το αρχείο είναι 204ΜΒ, και περιέχει 1.578 υπέρηχους μαστού, καταχωρισμένους σε τρείς κατηγορίες: benign, malignant, normal.

1. Θα εκπαιδεύσουμε ένα βαθύ νευρωνικό δίκτυο με τα δεδομένα μας.
2. Θα χρησιμοποιήσουμε **Transfer Learning** χρησιμοποιώντας δύο μεγάλα νευρωνικά εκπαιδευμένα στο ImageNet: ``Resnet18``, και το ``Inception V3``.
3. Θα εκπαιδεύσουμε το ``Resnet18`` μόνο με τα δεδομένα μας, και θα δούμε πως δεν θα πετύχουμε την ίδια ακρίβεια. Αυτό πιστοποιεί πως πράγματι, η εκπαίδευση πάνω στα δεδομένα του ImageNet, αν και είναι ποιοτικά εντελώς διαφορετικές, μας προσφέρουν ένα πλεονέκτημα.
4. Στην πορεία, θα μάθουμε και για Data Augmentation.

```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```

In [ ]:
from __future__ import print_function, division
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
import torchvision
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
import time
import os
import copy
import cv2

import requests
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import math
from sklearn.model_selection import train_test_split

plt.ion()   # interactive mode


In [ ]:
# Set the random seed
import torch
torch.manual_seed(42)

In [ ]:
# set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Τα Δεδομένα

[Μπορείτε να κατεβάσετε τα δεδομένα από αυτό το λινκ](https://drive.google.com/file/d/11aYgLvsPjAGVoDjY1r2deIWhIZN8knTD/view?usp=sharing).

Αφού τα κατεβάσετε, να τα βάλετε στο Google Drive σας.

Στο δικό μου Google Drive, βρίσκονται στον φάκελο:
```
'/content/drive/MyDrive/Colab Notebooks/YouTube-Data-Sets/BUS_dataset.zip'
```

Οι επόμενες δύο εντολές:
1. Δίνουν πρόσβαση στο Notebook σας να μπορέσει να δεί το Google Drive σας, και
2. Δίνουν την εντολή να γίνει ``unzipped`` το αρχείο με τα δεδομένα που χρεαιζόμαστε.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Data from this Kaggle page: https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset
!unzip '/content/drive/MyDrive/Colab Notebooks/YouTube-Data-Sets/BUS_dataset.zip' > /dev/null 2>&1



Κοιτάξτε τα δεδομένα -- βλέπουμε πως είναι οργανωμένα σε τρείς φακέλους, "benign", "malignant" και "normal" που αντιστοιχούν στις τρείς κατηγορίες.

Εάν δείτε μέσα στους φακέλους, μπορείτε να δείτε πως υπάρχουν εικόνες (υπέρηχοι) με ονομασία, πχ, "benign (10).png". Υπάρχουν επίσης αρχεία που λέγονται, πχ, "benign (10)_mask.png". Ανοίξτε ένα να το δείτε. Αυτή είναι πρόσθετη πληροφορία που θα μπορούσε να χρησιμοποιηθεί. Εμείς δεν θα την χρησιμοποιήσουμε. Οπότε πρέπει να τα σβήσουμε αυτά τα συγκεκριμένα αρχεία.

In [ ]:
# The data has files that are labeled as masks. We delete these:

# Base directory containing the subfolders
base_dir = '/content/Dataset_BUSI_with_GT'

# Loop through each subdirectory in the base directory
for sub_dir in os.listdir(base_dir):
    # Full path to the subdirectory
    full_path = os.path.join(base_dir, sub_dir)

    # Check if the current path is indeed a directory
    if os.path.isdir(full_path):
        # List all files in the subdirectory
        for file_name in os.listdir(full_path):
            # Check if 'mask' is in the file name
            if 'mask' in file_name:
                # Full path to the file
                file_path = os.path.join(full_path, file_name)
                # Remove the file
                os.remove(file_path)
                # print(f"Deleted {file_path}")


### Κοιτάμε τα δεδομένα (πάντα)



Παρατηρούμε πως κάποιες από τις εικόνες περιέχουν και κάποια γράμματα. Οποιαδήποτε τέτοια στοιχεία έχουν τα δεδομένα μας, απαιτούν πολύ προσοχή. Δεν θέλουμε το νευρωνικό μας δίκτυο να μάθει να χρησιμοποιεί τέτοια στοιχεία, που μπορεί να μην υπάρχουν στα μελλοντικά δείγματα.

Για τους σκοπούς μας (να μάθουμε τι είναι το Transfer Learning) όμως, αυτό το σετ δεδομένων αρκεί όπως είναι, και δεν μας απασχολούν πιθανά θέματα με τις εικόνες.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import random

folder_name = '/content/Dataset_BUSI_with_GT'
file_names = ['benign', 'malignant', 'normal']

def display_class_images(folder_name, k):
    """
    Displays k images from each class subfolder within the specified folder.
    """
    for file in os.listdir(folder_name):
        path = os.path.join(folder_name, file)
        fig, axes = plt.subplots(1, k, figsize=(15, 5))
        images = os.listdir(path)
        for x in range(k):
            img = cv2.imread(os.path.join(path, images[x]), cv2.IMREAD_GRAYSCALE)
            axes[x].imshow(img, cmap='gray')
            axes[x].set_title(f"Image {x+1} from {file} category")
            axes[x].axis('off')  # Turn off the axis for each subplot
        plt.suptitle(file, fontsize=26)
        plt.tight_layout()
        plt.show()

# Example usage:
k = 5  # Number of images to display from each class
display_class_images(folder_name, k)


# Το Πρόβλημα Είναι Δύσκολο

Για να πειστούμε πως είναι δύσκολο το πρόβλημα, διαλέγουμε κάποιες τυχαίες εικόνες από κάθε κατηγορία, αποκρύπτοντας όμως την προέλευση (την κατηγορία).

Μπορείτε χωρίς να δείτε να απάντηση, να λύσετε το πρόβλημα της ταξινόμησης;

Πρώτα γράφουμε 3 συναρτήσεις:

1. Η πρώτη διαλέγει τυχαία κάποιες εικόνες.
2. Η δεύτερη δίχνει τις εικόνες (τους υπέρηχους) χωρίς να αποκαλύψει την κατηγορία.
3. Η τρίτη δίχνει τις εικόνες μαζί με την απάντηση (την κατηγορία).

In [ ]:
def select_random_indices(folder_name, m):
    """
    Selects m random indices from the dataset.
    """
    images_paths = []

    for file in os.listdir(folder_name):
        path = os.path.join(folder_name, file)
        for img in os.listdir(path):
            images_paths.append(os.path.join(path, img))

    # Randomly select indices
    selected_indices = random.sample(range(len(images_paths)), m)
    return selected_indices

def display_random_images(folder_name, selected_indices, images_per_row=5):
    """
    Displays the selected images without labels.
    """
    images_paths = []

    for file in os.listdir(folder_name):
        path = os.path.join(folder_name, file)
        for img in os.listdir(path):
            images_paths.append(os.path.join(path, img))

    # Determine grid layout
    m = len(selected_indices)
    rows = (m + images_per_row - 1) // images_per_row
    fig, axes = plt.subplots(rows, images_per_row, figsize=(images_per_row * 3, rows * 3))

    axes = axes.flatten()

    for i, idx in enumerate(selected_indices):
        img = cv2.imread(images_paths[idx], cv2.IMREAD_GRAYSCALE)
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')

    for j in range(len(selected_indices), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

def display_random_images_with_labels(folder_name, selected_indices, images_per_row=5):
    """
    Displays the selected images with their labels written above.
    """
    images_paths = []
    labels = []

    for file in os.listdir(folder_name):
        path = os.path.join(folder_name, file)
        for img in os.listdir(path):
            images_paths.append(os.path.join(path, img))
            labels.append(file)

    # Determine grid layout
    m = len(selected_indices)
    rows = (m + images_per_row - 1) // images_per_row
    fig, axes = plt.subplots(rows, images_per_row, figsize=(images_per_row * 3, rows * 3))

    axes = axes.flatten()

    for i, idx in enumerate(selected_indices):
        img = cv2.imread(images_paths[idx], cv2.IMREAD_GRAYSCALE)
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(labels[idx], fontsize=10)

    for j in range(len(selected_indices), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()


### Διαλέγουμε Τυχαία

Διαλέγουμε $m = 20$ υπέρηχους.
Προσπαθήστε να αποφασήσετε εάν ο κάθε υπέρηχος είναι "benign", "malignant", ή "normal".

In [ ]:
# Select random indices
m = 20  # Number of images to display
indices = select_random_indices(folder_name, m)

# Display the images without labels
display_random_images(folder_name, indices, images_per_row=4)



### Οι Απαντήσεις

Για να δείτε τις απαντήσεις, τρέξτε τον παρακάτω κώδικα.

In [ ]:
# Display the same images with labels
display_random_images_with_labels(folder_name, indices, images_per_row=4)

# Ετοιμάζουμε τα δεδομένα

1. Φτιάχνουμε τα ``transforms``.
2. Χωρίζουμε σε δεδομένα εκπαίδευσης και δεδομένα εκτίμησης.
3. Δημιουργούμε τα Data Loaders

### Transforms & **Data Augmentation**

Χρησιμοποιούμε τα ``transform`` με πιο ουσιαστικό τρόπο:
* ``transforms.Grayscale``: χρησιμοποιείται όταν τα input είναι μαυρόασπρα (Grayscale).
* ``transforms.Resize(256)``: αλλάζει το μέγεθος της εικόνας ώστε η μικρότερη από τις δύο διαστάσεις να έχει 256 πίξελ.
* ``transforms.CenterCrop(224)``: κρατάει από την εικόνα το κεντρικό τετράγωνο 224 x 224.

Οι επόμενες τρείς εντολές υπάγονται στην γενικότερη τεχνική που λέγεται **Data Augmentation**. Το Data Augmentation (Αύξηση Δεδομένων) είναι μια τεχνική που χρησιμοποιείται για να ενισχύσει την ποικιλία των δεδομένων εκπαίδευσης σε προβλήματα μηχανικής όρασης. Είναι ιδιαίτερα σημαντικό να το κάνουμε αυτό, για διάφορους λόγους:

1. **Αντιμετώπιση της υπερμοντελοποίησης**.
Σε περιπτώσεις όπου το dataset είναι μικρό, το μοντέλο μπορεί να "μάθει" πολύ καλά τα δεδομένα εκπαίδευσης αλλά να αποτύχει σε νέα δεδομένα (overfitting). Το Data Augmentation προσθέτει ποικιλία στα δεδομένα δημιουργώντας νέες παραλλαγές (π.χ., περιστροφή εικόνας, αλλαγή φωτεινότητας, zoom, flip κ.λπ.), βοηθώντας το μοντέλο να γενικεύει καλύτερα.

2. **Περιορισμός Κόστους Συλλογής Δεδομένων**.
Η συλλογή δεδομένων είναι χρονοβόρα και ακριβή, και στην προκειμένη περίπτωση, έχει γίνει ήδη και δεν έχουμε την δυνατότητα να ζητήσουμε επιπλέον δεδομένα εκπαίδευσης. Με το Data Augmentation, μπορούμε να επεκτείνουμε το υπάρχον dataset.

3. **Ανθεκτικότητα σε Πραγματικές Συνθήκες**.
Στις πραγματικές εφαρμογές, οι εικόνες μπορεί να εμφανίζονται υπό διαφορετικές γωνίες, φωτισμό ή θόρυβο. Το Data Augmentation βοηθά το μοντέλο να είναι ανθεκτικό σε τέτοιες παραλλαγές, προετοιμάζοντάς το να ανταποκριθεί καλύτερα σε τέτοιες καταστάσεις.

4. **Εξισορρόπηση Ανισόρροπων Δεδομένων**. Το συγκεκριμένο πρόβλημα δεν παρουσιάζεται στα δεδομένα μας, αλλά το αναφέρουμε.
Σε datasets όπου κάποιες κατηγορίες είναι λιγότερο αντιπροσωπευμένες (π.χ. φανταστείτε πως μόνο ένα πολύ μικρό ποσοστό των δεδομένων μας ήταν "malignant") το Data Augmentation μπορεί να δημιουργήσει επιπλέον δείγματα για αυτές τις κατηγορίες, βελτιώνοντας την ισορροπία και την ακρίβεια της εκπαίδευσης.

* ``transforms.RandomRotation``: περιστρέφει τις εικόνες με τυχαία γωνία (από μηδέν εώς 10 μοίρες).
* ``transforms.RandomHorizontalFlip``: αντανακλά οριζοντίως  κάποιες (τυχαία επιλεγμένες) εικόνες.
* ``transforms.RandomAffine``: εφαρμόζει τυχαία μετατόπιση και αλλαγή κλίμακας.

In [ ]:
# Now define the data transformations that we need
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485], std=[0.229])  # Assuming grayscale
])


In [ ]:
# Path to your dataset
dataset_path = '/content/Dataset_BUSI_with_GT'

# Create a dataset loading utility that automatically applies the transforms
dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
class_names = dataset.classes
class_names

### Train test split

Χωρίζουμε τις εικόνες σε δεδομένα εκπαίδευσης και δεδομένα εκτίμησης.

In [ ]:
test_split_size = int(0.25 * len(dataset))
train_split_size = len(dataset) - test_split_size

np.random.seed(42)
train_dataset, test_dataset = random_split(dataset, [train_split_size, test_split_size])


### Create dataloaders

Τα Data Loaders τα έχουμε δεί πολλές φορές στα προηγούμενα Notebook.

In [ ]:
torch.manual_seed(42)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


## Εκπαιδεύουμε ένα CNN

Χτίζουμε και εκπαιδεύουμε ένα νευρωνικό δίκτυο. Χρησιμοποιούμε 4 ειδών επιπέδων:

* ``nn.Conv2d``
* ``nn.MaaxPool2d``
* ``nn.ReLU``
* ``nn.Linear``

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Breast_Cancer_CNN(nn.Module):
    def __init__(self, img_sz):
        super(Breast_Cancer_CNN, self).__init__()
        # Assuming img_sz is the size of the input image and images are grayscale
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # input channels, output channels, kernel size
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # kernel size, stride
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

        # Calculate the size of the output from the last convolutional layer
        output_size = img_sz // 4  # Divided by 4 because of two pooling layers each reducing size by half
        self.fc1 = nn.Linear(64 * output_size * output_size, 64)  # output channels * output width * output height
        self.fc2 = nn.Linear(64, 3)  # Assuming 3 classes

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv3(x)
        x = self.relu(x)
        x = torch.flatten(x, 1)  # Flatten all dimensions except batch
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        #return F.softmax(x, dim=1)  # Apply softmax at the output
        return x

# Example of creating the model
img_sz = 224  # Define the image size here
model = Breast_Cancer_CNN(img_sz)
print(model)


In [ ]:
# Πόσες παραμέτρους έχει το μοντέλο μας
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Εκπαίδευση

Πρώτα δίνουμε τον κώδικα που ορίζει την εντολή για εκπαίδευση. Είναι ακριβώς ο ίδιος που έχουμε χρησιμοποιήσει και στο προηγούμενο Notebook. Ορίζει την συνάρτηση:
```
train(model, train_loader, test_loader, optimizer, epochs)
```

In [ ]:
from torch import optim

import torch
import copy
import torch.nn.functional as F
import matplotlib.pyplot as plt

def train(model, train_loader, test_loader, optimizer, epochs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)


    train_losses = []
    test_losses = []
    test_accuracies = []  # List to store accuracy for each epoch
    best_accuracy = 0  # Best accuracy found
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()  # Update the learning rate

        train_losses.append(total_loss / len(train_loader))

        model.eval()
        total_test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_test_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        epoch_loss = total_test_loss / len(test_loader)
        epoch_accuracy = correct / total
        test_losses.append(epoch_loss)
        test_accuracies.append(epoch_accuracy)

        if epoch_accuracy > best_accuracy:
            best_accuracy = epoch_accuracy
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        print(f"Epoch {epoch+1}/{epochs}, Training Loss: {train_losses[-1]}, Testing Loss: {test_losses[-1]}, Testing Accuracy: {epoch_accuracy:.4f}")

    # Load best model weights
    model.load_state_dict(best_model_wts)
    print(f"Loaded the best model from epoch {best_epoch} with Testing Accuracy: {best_accuracy:.4f}")

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs+1), train_losses, label='Training Loss')
    plt.plot(range(1, epochs+1), test_losses, label='Testing Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs+1), test_accuracies, label='Testing Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.show()

    #return train_losses, test_losses, test_accuracies, best_accuracy


def evaluate(model, data_loader, device):
    torch.manual_seed(42)
    model = model.to(device)
    model.eval()  # Set model to evaluate mode
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

## Πόσες παραμέτρους;

Πόσες παραμέτρους έχει το μοντέλο μας;

In [ ]:
torch.manual_seed(42)
model = Breast_Cancer_CNN(img_sz=224)
count_trainable_parameters(model)

In [ ]:
torch.manual_seed(42)
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
epochs = 30
train(model, train_loader, test_loader, optimizer, epochs)


In [ ]:
evaluate(model, test_loader, device)

## 60% Ακρίβεια

Φαίνεται πως η επίδοση του μοντέλου έχει ισορροπήσει -- έχουμε φτάσει το όριο απόδοσης του μοντέλου μας.

### **Άσκηση**:
Θα μπορούσαμε να δοκιμάσουμε άλλο (πιο βαθύ, πιο μεγάλο) δίκτυο. **Αυτό το αφήνουμε σε σας**. Μπορείτε να δοκιμάσετε παραπάνω επίπεδα, και επίσης άλλα επίπεδα που έχουμε δει αλλά δεν χρησιμοποιήσαμε στον ορισμό του νευρωνικού μας, για παράδειγμα, ``BatchNorm`` και ``Dropout`` (δείτε για παράδειγμα, το[ Notebook αυτό](https://colab.research.google.com/drive/1PMhVak3v8e5nCcvQEEGvBNdcLalYcCeW?usp=sharing)).


# Transfer Learning

Θα χρησιμοποιήσουμε την τεχνική του Transfer Learning.

1. Κατεβάζουμε ένα ισχυρό -- βαθύ, με πάρα πολλές παραμέτρους, εκπαιδευμένο στις 14Μ εικόνες του Imagenet.
* Τώρα έχουμε ένα νευρωνικό δίκτυο που δεν έχει δεί κανέναν υπέρηχο, αλλά έχει δει εκατομμύρια άλλες εικόνες και ξέρει να τις ταξινομεί.
2. Προσθέτουμε ένα καινούργιο τελευταίο επίπεδο σε αυτό το νευρωνικό δίκτυο. Το τελευταίο επίπεδο έχει σχετικά λίγες παραμέτρους.
3. Εκπαιδεύουμε αυτές τις παραμέτρους, χρησιμοποιώντας τα δεδομένα που κατεβάσαμε.
4. Εκπαιδεύουμε τώρα όλες τις παραμέτρους μαζί.

Θα δούμε πως αυτή η τεχνική, εάν την εφαρμόσουμε σωστά, έχει καλή επιτυχία.

# Transfer Learning

Τώρα θα χρησιμοποιήσουμε την τεχνική του Transfer Learning.

Θα το δοκιμάσουμε με δύο διαφορετικά μοντέλο, εκπαιδευμένα στις 14.000.000 εικόνες του ImageNET:

1. Resnet18 -- ένα νευρωνικό δίκτυο με 18 επίπεδα (το χρησιμοποιήσαμε στο [προηγούμενο Notebook](https://colab.research.google.com/drive/1tnIQoViTVbT-7XiJa791h492B_WuNw4t?usp=sharing) για την CIFAR-10), και
2. InceptionV3 -- ένα νευρωνικό δίκτυο με 48 επίπεδα.



# Resnet18

Το Resnet18 περιέχει 17 convolutional επίπεδα, και ένα τελευταίο γραμμικό fully-connected επίπεδο. Λέγεται "Resnet" από την λέξη "residual". Όπως δίχνει και η εικόνα, κάθε input περνάει διαδοχικά από κάθε επίπεδο, αλλά υπάρχουν επιπλέον σύνδεσμοι που επιτρέπουν πιο άμεση πρόσβαση στα τελευταία επίπεδα.

<img src="https://www.researchgate.net/profile/Sajid-Iqbal-13/publication/336642248/figure/fig1/AS:839151377203201@1577080687133/Original-ResNet-18-Architecture.png" width=1024px/>

## Grayscale to RGB

Οι υπέρηχοι είναι grayscale -- δηλαδή μαυρόασπροι με μόνο ένα channel. Πρέπει να τους βάλουμε σε μορφή RGB με τρία κανάλια, για να ταιριάζουν με το Resnet18.

Αυτό θα το καρτορθώσουμε με τα ``transform``. Η διαφορά με τον προηγούμενο ορισμό του ``transform`` είναι η γραμμή:
```
transforms.Grayscale(num_output_channels=3)
```


In [ ]:
from torchvision import transforms
import torch

# Define the data transformations that we need
transformRGB = transforms.Compose([
    # Assuming your images might not be strictly 1-channel grayscale; if they are, you can remove this
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.485, 0.485], std=[0.229, 0.229, 0.229])  # Adjusted normalization
])

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# We are using ImageFolder
datasetRGB = ImageFolder(root=dataset_path, transform=transformRGB)


### Χωρίζουμε σε δεδομένα εκπαίδευσης και εκτίμησης

και δημιουργούμε τα καινούργια Data Loaders.

In [ ]:
# Define the size of the test set
test_split_size = int(0.25 * len(dataset))
train_split_size = len(dataset) - test_split_size

# Randomly split the dataset into train and test
train_datasetRGB, test_datasetRGB = random_split(datasetRGB, [train_split_size, test_split_size])


In [ ]:
# the dataloaders
train_loaderRGB = DataLoader(train_datasetRGB, batch_size=32, shuffle=True)
test_loaderRGB = DataLoader(test_datasetRGB, batch_size=32, shuffle=False)

In [ ]:
import matplotlib.pyplot as plt

def show_images(dataloader, classes, k=5):
    """
    Displays k images from the provided DataLoader with their corresponding labels.

    Args:
    dataloader (torch.utils.data.DataLoader): The DataLoader to sample images from.
    classes (list): List of class names corresponding to label indices.
    k (int): Number of images to display.
    """
    # Set the number of images you want to display at once
    dataiter = iter(dataloader)
    images, labels = next(dataiter)  # Get one batch of images and labels

    plt.figure(figsize=(10, 2 * k))
    for idx in range(k):
        ax = plt.subplot(1, k, idx + 1)
        img = images[idx].numpy().transpose((1, 2, 0))  # Convert from PyTorch tensor to NumPy array and transpose
        img = (img - img.min()) / (img.max() - img.min())  # Normalize to [0, 1] for visualization

        plt.imshow(img)
        plt.title(classes[labels[idx]])
        plt.axis('off')
    plt.show()

# Example usage:
# Assuming 'train_loader' is your DataLoader and 'class_names' is a list of class names
show_images(train_loaderRGB, class_names, k=7)

## Resnet18

Κατεβάζουμε προεκπαιδευμένο Resnet18, και αλλάζουμε το τελευταίο επίπεδο. Θυμόμαστε από το [προηγούμενο Notebook](https://colab.research.google.com/drive/1tnIQoViTVbT-7XiJa791h492B_WuNw4t?usp=sharing) πως για να το κάνουμε αυτό, πρέπει να μάθουμε πως είναι ονομασμένο, και πόσα input έχει το τελευταίο επίπεδο.

Με την εντολή:
```
layer_names = [name for name, _ in model.named_modules()]
for name in layer_names:
    print(name)
```
βλέπουμε το όνομα κάθε επιπέδου. Το τελευταίο επίπεδο λέγεται ``fc`` οπότε ``resnet18.fc`` εάν το μοντέλο μας το έχουμε ονομάσει ``resnet18``.

Τον αριθμό των input features του επιπέδου ``resnet18.mc`` τον βρίσκουμε με την εντολή
```
num_ftrs = resnet18.fc.in_features
```

In [ ]:
# Resnet
num_classes = 3

# Δείτε εδώ για άλλα μοντέλα https://pytorch.org/vision/stable/models.html
resnet18 = models.resnet18(weights = models.ResNet18_Weights.DEFAULT)

num_ftrs = resnet18.fc.in_features

torch.manual_seed(42)
resnet18.fc = nn.Linear(num_ftrs, num_classes)

# στέλνσουμε στο device (GPU εάν υπάρχει διαθέσιμο)
resnet18 = resnet18.to(device)

## Πρώτα εκπαιδεύουμε το τελευταίο επίπεδο

Παγώνουμε όλες τις παραμέτρους του μοντέλου εκτός από αυτές στο τελευταίο επίπεδο που μόλις προσθέσαμε.

In [ ]:
# Freeze all layers except the final fully connected layer
for param in resnet18.parameters():
    param.requires_grad = False

# Unfreeze the final layer
for param in resnet18.fc.parameters():
    param.requires_grad = True

In [ ]:
count_trainable_parameters(resnet18)

In [ ]:
torch.manual_seed(42)
optimizer = optim.Adam(resnet18.fc.parameters(), lr=0.0001, weight_decay=1e-4)
epochs = 40
train(resnet18, train_loaderRGB, test_loaderRGB, optimizer, epochs)

### Δεν βλέπουμε ακόμα κάποια βελτίωση

Μετά από 40 epochs εκπαιδεύοντας μόνο το τελευταίο επίπεδο (δηλαδή, μόνο 1.539 παραμετρους) η ακρίβεια που πετυχαίνουμε δεν υπερβαίνει την ακρίβεια που πετύχαμε χωρίς Transfer Learning.

### Τώρα εκπαιδεύουμε όλα τα επίπεδα

Εκπαιδεύουμε όλα τα επίπεδα.

In [ ]:
torch.manual_seed(42)

# Unfreeze all layers
for param in resnet18.parameters():
    param.requires_grad = True

count_trainable_parameters(resnet18)

In [ ]:
torch.manual_seed(42)

# Re-define the optimizer with a lower learning rate for fine-tuning
optimizer = optim.Adam(resnet18.parameters(), lr=0.0001, weight_decay=1e-4)

# Continue training with all layers unfrozen
additional_epochs = 40
train(resnet18, train_loaderRGB, test_loaderRGB, optimizer, additional_epochs)


## Σχεδόν 90% ακρίβεια

Πολύ καλά τα πήγαμε με το Transfer learning από την Resnet18.

# Inception V3

Μπορούμε να δοκιμάσουμε την ίδια ιδέα του Transfer Learning, με άλλα προ-εκπαιδευμένα δίκτυα, για παράδειγμα:
1. VGG16
2. Resnet51
3. Inception V3
4. Δείτε και εδώ για άλλα μοντέλα https://pytorch.org/vision/stable/models.html


Για όλα αυτά, η βασική ιδέα είναι ίδια, αλλά μπορεί να υπάρχουν μικρές αλλά απαραίτητες παραλαγές στον κώδικα. Θα δούμε ένα παράδειγμα με το Inception V3. Θα αναγκαστούμε να αλλάξουμε κάποια πράγματα σε σχέση με το Resnet18. Η βασική ιδέα και λογική του Transfer Learning ισχύει, όμως, και έτσι ακολουθούμε την ίδια λογική που είχαμε και παραπάνω.

In [ ]:
from torch import nn

#model = torch.hub.load('pytorch/vision:v0.10.0', 'inception_v3', pretrained=True)
inception = models.inception_v3(weights='Inception_V3_Weights.DEFAULT', aux_logits=True)

# Modify the final layers of the model
num_features = inception.fc.in_features  # Get the number of inputs for the last linear layer

inception.fc = nn.Linear(num_features, num_classes)
inception.AuxLogits.fc = nn.Linear(768, num_classes)  # Adjusting the auxiliary classifier

# Now send to device
inception = inception.to(device)

### Ετοιμάζουμε τα δεδομένα

Αυτό το νευρωνικό δίκτυο είναι εκπαιδευμένο σε εικόνες μεγέθους 299 x 299. Οπότε πρέπει να αλλάξουμε το μέγεθος των εικόνων μας ωστε να ταιριάζει με αυτό.


### Transforms

In [ ]:
# Define the data transformations that we need
transformINC = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(299),
    transforms.CenterCrop(299),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Adjusted normalization
])

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader



### Data Loaders

Χωρίζουμε σε δεδομένα εκπαίδευσης και εκτίμησης, και φτιάχνουμε τα Data Loaders.

In [ ]:
datasetINC = ImageFolder(root=dataset_path, transform=transformINC)

test_split_size = int(0.25 * len(datasetINC))
train_split_size = len(datasetINC) - test_split_size

# Randomly split the dataset into train and test
train_datasetINC, test_datasetINC = random_split(datasetINC, [train_split_size, test_split_size])

# the dataloaders
train_loaderINC = DataLoader(train_datasetINC, batch_size=32, shuffle=True)
test_loaderINC = DataLoader(test_datasetINC, batch_size=32, shuffle=False)

### Εκπαιδεύουμε το τελευταίο επίπεδο

* Παγώνουμε όλες τις παραμέτρους εκτός από αυτές στο τελευταίο επίπεδο.

In [ ]:
# Freeze all layers except the final fully connected layer
for param in inception.parameters():
    param.requires_grad = False

# Unfreeze the final layer
for param in inception.fc.parameters():
    param.requires_grad = True

In [ ]:
count_trainable_parameters(inception)

In [ ]:
def inc_train(model, train_loader, test_loader, optimizer, epochs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    is_inception = model.__class__.__name__ == "Inception3"

    train_losses = []
    test_losses = []
    test_accuracies = []
    best_accuracy = 0
    best_model_wts = copy.deepcopy(model.state_dict())

    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)



    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(images)
            if is_inception and hasattr(outputs, 'logits'):
                # Inception model with auxiliary outputs
                loss1 = criterion(outputs.logits, labels)
                loss2 = criterion(outputs.aux_logits, labels)
                loss = loss1 + 0.4 * loss2  # Combining main and auxiliary loss
            else:
                # General model output handling
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()  # Learning rate scheduler step

        train_losses.append(total_loss / len(train_loader))

        # Evaluation phase
        model.eval()
        total_test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                if is_inception and hasattr(outputs, 'logits'):
                    outputs = outputs.logits  # Only use the main output for evaluation
                loss = criterion(outputs, labels)
                total_test_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        epoch_loss = total_test_loss / len(test_loader)
        epoch_accuracy = correct / total
        test_losses.append(epoch_loss)
        test_accuracies.append(epoch_accuracy)

        if epoch_accuracy > best_accuracy:
            best_accuracy = epoch_accuracy
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        print(f"Epoch {epoch+1}/{epochs}, Training Loss: {train_losses[-1]}, Testing Loss: {test_losses[-1]}, Testing Accuracy: {epoch_accuracy:.4f}")

    # Load best model weights
    model.load_state_dict(best_model_wts)
    print(f"Loaded the best model from epoch {best_epoch} with Testing Accuracy: {best_accuracy:.4f}")

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs+1), train_losses, label='Training Loss')
    plt.plot(range(1, epochs+1), test_losses, label='Testing Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs+1), test_accuracies, label='Testing Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.show()

    return # train_losses, test_losses, test_accuracies, best_accuracy


### Optimize only last layer

In [ ]:
def inc_evaluate(model, data_loader, device):
    torch.manual_seed(42)
    model = model.to(device)
    model.eval()  # Set model to evaluate mode
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits  # Only use the main output for evaluation
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy


In [ ]:
torch.manual_seed(42)
optimizer = optim.Adam(inception.fc.parameters(), lr=0.0001, weight_decay=1e-4)
epochs = 40
inc_train(inception, train_loaderINC, test_loaderINC, optimizer, epochs)

### Unfreeze and optimize

In [ ]:
# Unfreeze all layers
for param in inception.parameters():
    param.requires_grad = True

count_trainable_parameters(inception)

In [ ]:

# Re-define the optimizer with a lower learning rate for fine-tuning
optimizer = optim.Adam(inception.parameters(), lr=0.0001, weight_decay=1e-4)

# Continue training with all layers unfrozen
torch.manual_seed(42)
additional_epochs = 40
inc_train(inception, train_loaderINC, test_loaderINC, optimizer, epochs)

### Παρόμοια ακρίβεια

Και με το InceptionV3 πετυχαίνουμε παρόμοια ακρίβεια.

## Υπερμοντελοποιούμε;

Υπάρχει μία διαφορά μεταξύ της ακρίβειας στα δεδομένα εκπαίδευσης (training data) και δεδομένα εκτίμησης (testing data). Αλλά η ακρίβεια στα δεδομένα εκτίμησης είναι αρκετά καλή.

In [ ]:
inc_evaluate(inception, train_loaderINC, device)

In [ ]:
count_trainable_parameters(inception)

## **Άσκηση**

Δοκιμάστε μόνοι σας την ιδέα του Transfer Learning με άλλα προεκπαιδευμένα νευρωνικά δίκτυα.

# Εκπαιδεύοντας το Resnet από το μηδέν

Θα προσπαθήσουμε να εκπαιδεύσουμε το Resnet18 από το μηδέν, δηλαδή, αρχικοποιώντας τυχαία τις παραμέτρους, αντί να αρχίζουμε από το μοντέλο που έχει προεκπαιδευτεί με τις 14.000.000 εικόνες του ImageNet.

Θέλουμε να δούμε εάν όντως μας βοηθάει αυτή η προεκπαίδευση.

Θα χρησιμοποιήσουμε πολύ παρόμοιες εντολές για να κατεβάσουμε το Resnet18 δίκτυο, αλλά μή-εκπαιδευμένο -- οι παράμετροι θα είναι τυχαία αρχικοποιημένες.

Εκπαιδεύουμε για 100+100+100 epochs. Θα δούμε πως δεν μπορούμε να πλησιάσουμε την επίδοση του Resnet18 με Transfer Learning. Σε τι αποδήδουμε την διαφορά ακρίβειας; Αφού το δίκτυο είναι το ίδιο, είναι ένδειξη πως όσο και διαφορετικά να είναι τα δεδομένα (οι εικόνες) του ImageNet σε σχέση με τους υπέρηχους, βοηθάν το δίκτυο να γενικεύση καλύτερα στα δεδομένα εκτίμησης.

In [ ]:
# Resnet
num_classes = 3

# φορτώνουμε μή-εκπαιδευμένο μοντέλο
new_resnet18 = models.resnet18()

num_ftrs = new_resnet18.fc.in_features

torch.manual_seed(42)
new_resnet18.fc = nn.Linear(num_ftrs, num_classes)

# Now send to device
new_resnet18 = new_resnet18.to(device)

In [ ]:
count_trainable_parameters(new_resnet18)

In [ ]:
torch.manual_seed(42)

# The optimizer
optimizer = optim.Adam(new_resnet18.parameters(), lr=0.0001, weight_decay=1e-4)

# Continue training with all layers unfrozen
epochs = 100
train(new_resnet18, train_loaderRGB, test_loaderRGB, optimizer, epochs)

In [ ]:
torch.manual_seed(42)

# The optimizer
optimizer = optim.Adam(new_resnet18.parameters(), lr=0.0001, weight_decay=1e-4)

# Continue training with all layers unfrozen
epochs = 100
train(new_resnet18, train_loaderRGB, test_loaderRGB, optimizer, epochs)

In [ ]:
torch.manual_seed(42)

# Re-define the optimizer with a lower learning rate for fine-tuning
optimizer = optim.Adam(new_resnet18.parameters(), lr=0.00001, weight_decay=1e-4)

# Continue training with all layers unfrozen
epochs = 100
train(new_resnet18, train_loaderRGB, test_loaderRGB, optimizer, epochs)